In [1]:
import numpy as np
from lookup_table import CaseNum2EdgeOffset, getCaseNum
import trimesh
import os
import time

In [2]:
def marching_cube(thres, cells):
    # vertices use dictionary to avoid duplicate axes
    vertex_array = {}
    face_array = []
    t1 = time.time()
    # -------------------TODO------------------ 
    # compute vertices and faces
    # vertices: [N, 3]
    # faces: [M, 3], e.g. np.array([[0,1,2]]) means a triangle composed of vertices[0], vertices[1] and vertices[2]
    # for-loop is allowed to reduce difficulty
    # -------------------TODO------------------ 
    xx, yy, zz = cells.shape
    vertex_index = 0
    for x in range(xx - 1):
        for y in range(yy - 1):
            for z in range(zz - 1):
                case_nums = getCaseNum(x, y, z, thres, cells)
                local_vertex_array = []
                for case_num in case_nums:
                    if case_num == -1:
                        continue
                    corner1 = np.array([x + CaseNum2EdgeOffset[case_num][0], 
                                        y + CaseNum2EdgeOffset[case_num][1],
                                        z + CaseNum2EdgeOffset[case_num][2]])
                    corner2 = np.array([x + CaseNum2EdgeOffset[case_num][3], 
                                        y + CaseNum2EdgeOffset[case_num][4],
                                        z + CaseNum2EdgeOffset[case_num][5]])
                    # linear interpolation
                    val1 = cells[*corner1]
                    val2 = cells[*corner2]
                    dx, dy, dz = corner2 - corner1
                    df = val2 - val1
                    new_dx = (val2 - thres) / df * dx
                    new_dy = (val2 - thres) / df * dy
                    new_dz = (val2 - thres) / df * dz
                    vertex = corner2 - np.array([new_dx, new_dy, new_dz])
                    vertex_key = tuple(np.round(vertex, 3)) # round to avoid floating point error
                    if vertex_key not in vertex_array:
                        vertex_array[vertex_key] = (vertex, vertex_index)   # store vertex and index
                        vertex_index += 1
                    local_vertex_array.append(vertex_array[vertex_key][1])
                    if len(local_vertex_array) == 3:
                        face_array.append(local_vertex_array.copy())
                        local_vertex_array.clear()
    t2 = time.time()
    print("\nTime taken by algorithm\n"+'-'*40+"\n{} s".format(t2-t1))
    vertex_array = list(vertex_array.keys())    # directly use round vertex coordinates (i.e., keys)
    return np.array(vertex_array), np.array(face_array)

In [3]:
# reconstruct these two animals
shape_name_lst = ['spot', 'bob']
for shape_name in shape_name_lst:
    data = np.load(os.path.join('data', shape_name + '_cell.npy'))
    verts, faces = marching_cube(0, data)
    print(verts.shape)
    print(faces.shape)
    mesh = trimesh.Trimesh(vertices=verts, faces=faces)
    mesh_txt = trimesh.exchange.obj.export_obj(mesh)
    with open(os.path.join('../results', shape_name + '.obj'),"w") as fp:
        fp.write(mesh_txt)


Time taken by algorithm
----------------------------------------
5.743916273117065 s
(6850, 3)
(13712, 3)

Time taken by algorithm
----------------------------------------
6.060064792633057 s
(8959, 3)
(17936, 3)


In [8]:
# visualize mesh
# mesh = trimesh.load_mesh('../results/bob.obj')
# mesh.show()

In [9]:
# visualize mesh
# mesh = trimesh.load_mesh('../results/spot.obj')
# mesh.show()